# Apache Spark Data Cleaning, Transformation and Aggregation using DataFrames

## Objective

The objective of this notebook is to understand Apache Spark fundamentals and perform data cleaning, transformation, filtering, aggregation, schema modification, and data processing using Spark DataFrames.

The notebook demonstrates:
- Spark DataFrame concepts
- DataFrame immutability
- Data cleaning techniques
- Data transformation operations
- Filtering operations
- Aggregation functions
- GroupBy operations
- Shuffle and wide transformations
- Schema modification
- Handling inconsistent data
- Complete data processing pipeline

## Import Required Libraries

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Create Spark Session

In [8]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("SparkDataProcessingPipeline") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


## Load Dataset

In [9]:
df = spark.read.csv(
    r"Sample - Superstore.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    quote='"',
    escape='"'
)

## Dataset Overview

In [4]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

df.show(5)

Rows: 9994
Columns: 21
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   B

## Dataset Schema

In [5]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



## Understanding Spark DataFrames

A Spark DataFrame is a distributed collection of data organized into rows and columns. It provides an easy and efficient way to process large datasets and supports SQL-like operations such as filtering, grouping, aggregation, and transformations.

## Understanding DataFrame Immutability

Spark DataFrames are immutable, meaning they cannot be modified after creation.

Operations such as filtering, dropping columns, renaming columns, or handling null values create a new DataFrame instead of changing the original one.

Benefits:
- Improved fault tolerance
- Better parallel processing
- Safer data handling
- Efficient Spark optimization

In [10]:
print("Original Columns:")
print(df.columns)

new_df = df.drop("Postal Code")

print("New DataFrame Columns:")
print(new_df.columns)

print("Original DataFrame Still Exists:")
print(df.columns)

Original Columns:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
New DataFrame Columns:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
Original DataFrame Still Exists:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


## Data Cleaning - Removing Duplicate Records

In [11]:
df_clean = df.dropDuplicates(["Order ID"])

print("Original Records:", df.count())
print("After Removing Duplicates:", df_clean.count())

Original Records: 9994
After Removing Duplicates: 5009


## Data Cleaning - Handling Null Values

In [12]:
df_clean = df_clean.na.fill({
    "City": "Unknown"
})

df_clean.show(5)

+------+--------------+----------+---------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|   Customer Name|    Segment|      Country|         City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+---------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|  2718|CA-2014-100006|  9/7/2014|9/13/2014|Standard Class|   DK-13375|     Dennis Kane|   Consumer|United States|New York City|  New York|      10024|  East|TEC-PH-10002075|     Technology|      Phones|   AT&T EL51110 D

## Data Transformation - Schema Modification

In [13]:
df_clean = df_clean.withColumn(
    "Sales",
    regexp_replace(col("Sales"), ",", "").cast("double")
)

## Data Transformation - Type Casting

In [14]:
df_clean = df_clean.withColumn(
    "Quantity",
    col("Quantity").cast("int")
)

df_clean = df_clean.withColumn(
    "Discount",
    col("Discount").cast("double")
)

df_clean.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = false)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



## Filtering Operations

In [12]:
#Filter by Region
west_df = df_clean.filter(
    col("Region") == "West"
)

west_df.show(5)

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|         City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|  6288|CA-2014-100090|  7/8/2014| 7/12/2014|Standard Class|   EB-13705|       Ed Braxton|  Corporate|United States|San Francisco|California|      94122|  West|FUR-TA-10003715|      Furniture|      Tables|Hon 2111 

In [13]:
#Filter by Category
furniture_df = df_clean.filter(
    col("Category") == "Furniture"
)

furniture_df.show(5)

+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------------+--------------+-----------+------+---------------+---------+------------+--------------------+-------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|  Segment|      Country|         City|         State|Postal Code|Region|     Product ID| Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------------+--------------+-----------+------+---------------+---------+------------+--------------------+-------+--------+--------+--------+
|  6288|CA-2014-100090|  7/8/2014| 7/12/2014|Standard Class|   EB-13705|        Ed Braxton|Corporate|United States|San Francisco|    California|      94122|  West|FUR-TA-10003715|Furniture|      Tables|Hon 2111 Invitati...|

In [14]:
#Multiple Conditions
filtered_df = df_clean.filter(
    (col("Region") == "West") &
    (col("Category") == "Technology")
)

filtered_df.show(5)

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+----------+-----------+------+---------------+----------+------------+--------------------+--------+--------+--------+-------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|         City|     State|Postal Code|Region|     Product ID|  Category|Sub-Category|        Product Name|   Sales|Quantity|Discount| Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+----------+-----------+------+---------------+----------+------------+--------------------+--------+--------+--------+-------+
|  9462|CA-2014-100867|10/19/2014|10/24/2014|Standard Class|   EH-14125|Eugene Hildebrand|Home Office|United States|     Lakewood|California|      90712|  West|TEC-PH-10004922|Technology|      Phones|RCA Visys Integra...| 321.552

## Aggregation Functions

In [15]:
df_clean.agg(
    count("*").alias("Total_Records"),
    sum("Sales").alias("Total_Sales"),
    avg("Sales").alias("Average_Sales"),
    min("Sales").alias("Minimum_Sales"),
    max("Sales").alias("Maximum_Sales")
).show()

+-------------+------------------+-----------------+-------------+-------------+
|Total_Records|       Total_Sales|    Average_Sales|Minimum_Sales|Maximum_Sales|
+-------------+------------------+-----------------+-------------+-------------+
|         5009|1099862.0727000025|219.5771756238775|        0.556|    11199.968|
+-------------+------------------+-----------------+-------------+-------------+



## GroupBy Operations

In [16]:
df_clean.groupBy("Category") \
    .agg(
        sum("Sales").alias("Total_Sales")
    ) \
    .show()

+---------------+-----------------+
|       Category|      Total_Sales|
+---------------+-----------------+
|Office Supplies|345716.4560000011|
|      Furniture|373504.7207000014|
|     Technology|380640.8960000004|
+---------------+-----------------+



In [17]:
#GroupBy with Condition
df_clean.groupBy("City") \
    .count() \
    .filter(col("count") > 100) \
    .show()

+-------------+-----+
|         City|count|
+-------------+-----+
| Philadelphia|  265|
|  Los Angeles|  384|
|San Francisco|  265|
|     Columbus|  111|
|      Chicago|  171|
|      Seattle|  212|
|New York City|  450|
|      Houston|  188|
+-------------+-----+



## Understanding Shuffle and Wide Transformations

A shuffle occurs when Spark redistributes data between partitions during operations such as groupBy(), join(), and aggregations.

Shuffle is considered a wide transformation because data must move across multiple partitions before processing.

Since shuffle requires additional memory and network communication, it is one of the most expensive Spark operations.

In [18]:
df_clean.groupBy("Region") \
    .agg(
        sum("Sales").alias("Total_Sales")
    ) \
    .show()

+-------+------------------+
| Region|       Total_Sales|
+-------+------------------+
|  South|190409.22599999997|
|Central|       246314.5152|
|   East| 330694.6260000002|
|   West| 332443.7055000011|
+-------+------------------+



## Handling Inconsistent Data

In [19]:
df_clean.select([
    count(
        when(
            col(c).isNull(),
            c
        )
    ).alias(c)
    for c in df_clean.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [20]:
#Remove Invalid Records
valid_df = df_clean.filter(
    col("City").isNotNull()
)

valid_df.show(5)

+------+--------------+----------+---------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|   Customer Name|    Segment|      Country|         City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+---------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|  2718|CA-2014-100006|  9/7/2014|9/13/2014|Standard Class|   DK-13375|     Dennis Kane|   Consumer|United States|New York City|  New York|      10024|  East|TEC-PH-10002075|     Technology|      Phones|   AT&T EL51110 D

## Complete Data Processing Pipeline

In [15]:
pipeline_df = (
    df
    .dropDuplicates(["Order ID"])
    .na.fill({"City": "Unknown"})
    .withColumn(
        "Sales",
        regexp_replace(col("Sales"), ",", "").cast("double")
    )
)

final_result = (
    pipeline_df
    .groupBy("Region")
    .agg(
        sum("Sales").alias("Total_Revenue"),
        avg("Sales").alias("Average_Sales")
    )
)

final_result.show()

+-------+------------------+------------------+
| Region|     Total_Revenue|     Average_Sales|
+-------+------------------+------------------+
|  South|190409.22599999997| 231.6413941605839|
|Central|       246314.5152|209.62937463829786|
|   East| 330694.6260000002| 236.0418458244113|
|   West| 332443.7055000011| 206.3586005586599|
+-------+------------------+------------------+



In [18]:
import os

os.makedirs("outputs", exist_ok=True)

final_result.toPandas().to_csv(
    "outputs/final_pipeline_result.csv",
    index=False
)

print("CSV Exported Successfully")

CSV Exported Successfully


## Brief Insights on Data Processing and Transformations

1. Duplicate records were removed using dropDuplicates().
2. Missing values were handled using na.fill().
3. Schema modifications were performed using type casting.
4. Filtering operations helped analyze specific subsets of data.
5. Aggregation functions generated meaningful business statistics.
6. groupBy() enabled category-wise and region-wise analysis.
7. Shuffle operations occurred during grouping operations.
8. A complete Spark pipeline was built by combining cleaning, transformation, and aggregation.

## Conclusion

This notebook demonstrated Spark DataFrame operations including cleaning, transformation, filtering, aggregation, grouping, schema modification, handling inconsistent data, and building a complete data processing pipeline. Spark's in-memory processing makes large-scale data analysis faster and more efficient than traditional approaches.

In [20]:
spark.stop()